# V3 Swarm Visualization

This notebook allows you to select an episode from the generated dataset and visualize the drone swarm's movement step-by-step in 3D.
It also visualizes the static obstacles and the final target formation positions.

**Requirements**: `pip install plotly ipywidgets`

In [1]:
import sys, torch
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display
sys.path.insert(0, '../src')
from dataloader import SplitDataset

## 1. Load Dataset
Load the dataset to access the generated episodes.

In [2]:
DATASET_PATH = r"../datasets/setpoint_V3_mixed_formations_val.pt"
try:
    dataset = SplitDataset(DATASET_PATH)
    print(f"Loaded {len(dataset)} frames.")
except Exception as e:
    print(f"Could not load dataset (maybe it's still generating?): {e}")

Loaded 200 frames.


## 2. Group by Episode
Organize the frames by their `episode_id` so we can easily select and play back a specific episode.

In [3]:
episodes = {}
for i in range(len(dataset)):
    graph = dataset[i]
    ep_id = graph.episode_id.item()
    if ep_id not in episodes:
        episodes[ep_id] = []
    episodes[ep_id].append(graph)

for ep_id in episodes:
    episodes[ep_id] = sorted(episodes[ep_id], key=lambda g: g.step_idx.item())

episode_ids = list(episodes.keys())
print(f"Found {len(episode_ids)} unique episodes. IDs: {episode_ids[:10]}...")

Found 1 unique episodes. IDs: [8]...


## 3. Interactive 3D Visualization
Run the cell below to load the interactive UI. Select an episode from the dropdown, and use the slider to scrub through the steps.

In [4]:
def create_cylinder(x, y, r, h=10.0):
    z = np.linspace(0, h, 10)
    theta = np.linspace(0, 2*np.pi, 20)
    theta_grid, z_grid = np.meshgrid(theta, z)
    x_grid = r * np.cos(theta_grid) + x
    y_grid = r * np.sin(theta_grid) + y
    return x_grid, y_grid, z_grid

def create_circle(x, y, r):
    theta = np.linspace(0, 2*np.pi, 50)
    return r * np.cos(theta) + x, r * np.sin(theta) + y

fig_3d = go.FigureWidget()
fig_3d.update_layout(
    title='Drone Swarm Episode Replay (3D)',
    uirevision='constant',
    scene=dict(
        xaxis=dict(title='X', range=[-10, 10]),
        yaxis=dict(title='Y', range=[-10, 10]),
        zaxis=dict(title='Z', range=[0, 10]),
        aspectmode='cube'
    ),
    margin=dict(l=0, r=0, b=0, t=40),
    height=500, width=500
)

fig_2d = go.FigureWidget()
fig_2d.update_layout(
    title='Drone Swarm Episode Replay (2D Top-Down)',
    uirevision='constant',
    xaxis=dict(title='X', range=[-10, 10], scaleanchor='y', scaleratio=1),
    yaxis=dict(title='Y', range=[-10, 10]),
    margin=dict(l=0, r=0, b=0, t=40),
    plot_bgcolor='white',
    height=500, width=500
)

ep_dropdown = widgets.Dropdown(options=episode_ids, description='Episode:')
step_slider = widgets.IntSlider(min=0, max=10, step=1, description='Step:')
play_button = widgets.Play(min=0, max=10, step=1, description='Play', interval=33)
widgets.jslink((play_button, 'value'), (step_slider, 'value'))

fullscreen_btn = widgets.ToggleButton(description='Expand / Fullscreen', icon='arrows-alt')
def on_fullscreen_change(change):
    if change['new']:
        fig_3d.layout.height = 800
        fig_2d.layout.height = 800
        fig_3d.layout.width = 800
        fig_2d.layout.width = 800
    else:
        fig_3d.layout.height = 500
        fig_2d.layout.height = 500
        fig_3d.layout.width = 500
        fig_2d.layout.width = 500
fullscreen_btn.observe(on_fullscreen_change, names='value')

def init_plot(ep_id):
    fig_3d.data = []
    fig_2d.data = []
    if not episodes: return
    g = episodes[ep_id][0]
    
    if hasattr(g, 'target_pos') and g.target_pos is not None:
        targets = g.target_pos.numpy()
        fig_3d.add_trace(go.Scatter3d(
            x=targets[:,0], y=targets[:,1], z=targets[:,3],
            mode='markers', marker=dict(size=6, color='blue', symbol='cross'),
            name='Targets'
        ))
        fig_2d.add_trace(go.Scatter(
            x=targets[:,0], y=targets[:,1],
            mode='markers', marker=dict(size=8, color='blue', symbol='cross'),
            name='Targets'
        ))
        
    if hasattr(g, 'obstacles') and g.obstacles is not None and len(g.obstacles) > 0:
        obs = g.obstacles.numpy()
        for i, o in enumerate(obs):
            x, y, r = o[0], o[1], o[2]
            xg, yg, zg = create_cylinder(x, y, r)
            fig_3d.add_trace(go.Surface(
                x=xg, y=yg, z=zg, opacity=0.2, colorscale='Reds', showscale=False,
                name=f'Obstacle {i}', hoverinfo='skip'
            ))
            cx, cy = create_circle(x, y, r)
            fig_2d.add_trace(go.Scatter(
                x=cx, y=cy, mode='lines', fill='toself', fillcolor='rgba(200,0,0,0.2)',
                line=dict(color='red'), name=f'Obstacle {i}', hoverinfo='skip'
            ))
            
    pos = g.pos.numpy()
    fig_3d.add_trace(go.Scatter3d(
        x=pos[:,0], y=pos[:,1], z=pos[:,2],
        mode='markers+text', marker=dict(size=8, color='green', symbol='circle'),
        text=[str(i) for i in range(len(pos))], textposition='top center',
        name='Drones'
    ))
    fig_2d.add_trace(go.Scatter(
        x=pos[:,0], y=pos[:,1],
        mode='markers+text', marker=dict(size=12, color='green', symbol='circle'),
        text=[str(i) for i in range(len(pos))], textposition='top center',
        name='Drones'
    ))

def update_plot(ep_id, step_idx):
    if not episodes: return
    graphs = episodes[ep_id]
    step_idx = min(step_idx, len(graphs) - 1)
    g = graphs[step_idx]
    pos = g.pos.numpy()
    
    # Calculate drone colors (red if scratching obstacle)
    colors = ['green'] * len(pos)
    if hasattr(g, 'obstacles') and g.obstacles is not None and len(g.obstacles) > 0:
        obs = g.obstacles.numpy()
        for i, (dx, dy, dz) in enumerate(pos):
            for ox, oy, orad in obs:
                dist = np.sqrt((dx - ox)**2 + (dy - oy)**2)
                if dist <= orad + 0.15: # 0.15 is the assumed drone collision radius
                    colors[i] = 'red'
                    break
    
    with fig_3d.batch_update():
        if len(fig_3d.data) > 0:
            fig_3d.data[-1].x, fig_3d.data[-1].y, fig_3d.data[-1].z = pos[:,0], pos[:,1], pos[:,2]
            fig_3d.data[-1].marker.color = colors
            
    with fig_2d.batch_update():
        if len(fig_2d.data) > 0:
            fig_2d.data[-1].x, fig_2d.data[-1].y = pos[:,0], pos[:,1]
            fig_2d.data[-1].marker.color = colors

def on_ep_change(change):
    ep_id = change['new']
    if ep_id in episodes:
        max_step = len(episodes[ep_id]) - 1
        step_slider.max = max_step
        play_button.max = max_step
        step_slider.value = 0
        init_plot(ep_id)
        update_plot(ep_id, 0)

def on_step_change(change): update_plot(ep_dropdown.value, change['new'])

ep_dropdown.observe(on_ep_change, names='value')
step_slider.observe(on_step_change, names='value')

if episode_ids:
    ep_id = episode_ids[0]
    max_step = len(episodes[ep_id]) - 1
    step_slider.max = max_step
    play_button.max = max_step
    init_plot(ep_id)
    update_plot(ep_id, 0)

controls = widgets.HBox([play_button, step_slider, fullscreen_btn])
plots = widgets.HBox([fig_2d, fig_3d])
display(widgets.VBox([ep_dropdown, controls, plots]))
